# Keypoint-MoSeq workflow scaffold with STORM

This notebook checks pose loading and serializable study configuration. The external KPMS adapter is pending; it does not emit placeholder states that could be mistaken for model predictions.

In [ ]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory


def find_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "packages" / "rainstorm-thesis" / "src" / "rainstorm_thesis").is_dir():
            return candidate
    raise RuntimeError("RAINSTORM project root not found")


PROJECT_ROOT = find_root(Path.cwd())
THESIS_SRC = PROJECT_ROOT / "packages" / "rainstorm-thesis" / "src"
for path in (PROJECT_ROOT, THESIS_SRC):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT

In [ ]:
from storm import FileArtifactStore, ModelRegistry, Study, StudySpec
from rainstorm_thesis import (
    ExternalSegmentationModel,
    PoseDatasetConfig,
    build_pose_dataset_loader,
    build_segmentation_study_spec,
    example_pose_file,
    load_pose_table,
)

pose_path = example_pose_file("NOR_TS_02*.h5")
pose_table = load_pose_table(pose_path, max_frames=1_000)
pose_table.describe().T.head()

In [ ]:
config = PoseDatasetConfig(pose_path=pose_path, max_frames=1_000)
data_ref = config.data_ref(identifier="nor-ts-02-kpms")
data_loader = build_pose_dataset_loader(config)
dataset = data_loader(data_ref)

dataset.inputs.shape, dataset.metadata["frame_count"]

In [ ]:
models = ModelRegistry()
models.register("kpms", ExternalSegmentationModel)

spec: StudySpec = build_segmentation_study_spec(
    study_id="rainstorm-kpms-storm-demo",
    data=data_ref,
    model_family="kpms",
    variants=("pose_base", "pose_ego", "pose_roi", "ego_roi"),
    n_states=10,
)

spec.to_dict()

In [ ]:
workspace = TemporaryDirectory(prefix="rainstorm-kpms-storm-")
study = Study(
    spec,
    data_loader=data_loader,
    models=models,
    artifacts=FileArtifactStore(Path(workspace.name) / "artifacts"),
)
results = study.run()

[(result.run_id, result.load_output().metadata) for result in results]

In [ ]:
workspace.cleanup()